In [36]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/amazon-product-reviews-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Qamar Hasan\.cache\kagglehub\datasets\yasserh\amazon-product-reviews-dataset\versions\1


In [37]:
import pandas as pd
import re
from sklearn.utils import resample
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib
from imblearn.over_sampling import SMOTE

import torch
from torch.utils.data import Dataset

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments

In [38]:
# Adjust the path based on your download
dataset_path = path + "/7817_1.csv"

# Load the CSV file
df = pd.read_csv(dataset_path)

# Take a look at columns
df.head()

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams
3,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I bought one of the first Paperwhites and have...,Love / Hate relationship,NaN,NaN,Dougal,NaN,NaN,205 grams
4,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I have to say upfront - I don't like coroporat...,I LOVE IT,NaN,NaN,Miljan David Tanic,NaN,NaN,205 grams


In [39]:
df.columns

Index(['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded',
       'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer',
       'manufacturerNumber', 'name', 'prices', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes',
       'upc', 'weight'],
      dtype='object')

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1597 entries, 0 to 1596
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    1597 non-null   object 
 1   asins                 1597 non-null   object 
 2   brand                 1597 non-null   object 
 3   categories            1597 non-null   object 
 4   colors                774 non-null    object 
 5   dateAdded             1597 non-null   object 
 6   dateUpdated           1597 non-null   object 
 7   dimension             565 non-null    object 
 8   ean                   898 non-null    float64
 9   keys                  1597 non-null   object 
 10  manufacturer          965 non-null    object 
 11  manufacturerNumber    902 non-null    object 
 12  name                  1597 non-null   object 
 13  prices                1597 non-null   object 
 14  reviews.date          1217 non-null   object 
 15  reviews.doRecommend  

In [41]:
df = df[['reviews.rating', 'reviews.text']]

In [42]:
df

,reviews.rating,reviews.text
0,5.0,I initially had trouble deciding between the p...
1,5.0,Allow me to preface this with a little history...
2,4.0,I am enjoying it so far. Great for reading. Ha...
3,5.0,I bought one of the first Paperwhites and have...
4,5.0,I have to say upfront - I don't like coroporat...
...,...,...
1592,3.0,This is not the same remote that I got for my ...
1593,1.0,I have had to change the batteries in this rem...
1594,1.0,"Remote did not activate, nor did it connect to..."
1595,3.0,It does the job but is super over priced. I fe...


In [43]:
df['reviews.rating'].isna().sum()

np.int64(420)

In [44]:
df = df.dropna(subset=['reviews.rating'])


In [45]:
print(df.shape)  # Should be 1597 - 420 = 1177 rows
print(df['reviews.rating'].isna().sum())  # Should be 0

(1177, 2)
0


In [46]:
def label_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

df['sentiment'] = df['reviews.rating'].apply(label_sentiment)

C:\Users\Qamar Hasan\AppData\Local\Temp\ipykernel_15380\2122296229.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['reviews.rating'].apply(label_sentiment)


In [47]:
print("Original sentiment distribution:\n", df['sentiment'].value_counts())

Original sentiment distribution:
 sentiment
positive    977
neutral     124
negative     76
Name: count, dtype: int64


In [48]:
df_pos = df[df['sentiment'] == 'positive']
df_neu = df[df['sentiment'] == 'neutral']
df_neg = df[df['sentiment'] == 'negative']

In [49]:
# Downsample positives to 2x neutral
n_pos = min(len(df_pos), len(df_neu) * 2)
df_pos_down = resample(df_pos, replace=False, n_samples=n_pos, random_state=42)


In [50]:
df_neg_up = resample(df_neg, replace=True, n_samples=len(df_neu), random_state=42)


In [51]:
df_neu_up = resample(df_neu, replace=True, n_samples=len(df_neu)*2, random_state=42)

In [52]:
# Combine all
df_balanced = pd.concat([df_pos_down, df_neu_up, df_neg_up])

print("\nBalanced distribution:\n", df_balanced['sentiment'].value_counts())



Balanced distribution:
 sentiment
positive    248
neutral     248
negative    124
Name: count, dtype: int64


In [53]:
# -------------------------
# Step 5: Clean text
# -------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_balanced['clean_text'] = df_balanced['reviews.text'].apply(clean_text)

In [54]:
# -------------------------
# Step 6: TF-IDF
# -------------------------
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(df_balanced['clean_text'])
y = df_balanced['sentiment']

In [55]:
# -------------------------
# Step 7: Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [57]:
X_train_dense = X_train.toarray()

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_dense, y_train)

print("After SMOTE, class distribution:\n", pd.Series(y_train_res).value_counts())

After SMOTE, class distribution:
 sentiment
positive    199
negative    199
neutral     199
Name: count, dtype: int64


In [63]:
# -------------------------
# Step 8: Train Logistic Regression with class weights
# -------------------------
model = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    solver="saga",
    multi_class="multinomial",
    random_state=42
)
model.fit(X_train, y_train)

C:\Users\Qamar Hasan\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'saga'
,max_iter,500
,multi_class,'multinomial'


In [64]:
# -------------------------
# Step 9: Evaluate
# -------------------------
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, digits=4))

              precision    recall  f1-score   support

    negative     1.0000    0.8000    0.8889        25
     neutral     0.9535    0.8200    0.8817        50
    positive     0.7705    0.9592    0.8545        49

    accuracy                         0.8710       124
   macro avg     0.9080    0.8597    0.8751       124
weighted avg     0.8906    0.8710    0.8724       124



In [69]:
def test_sentiment(text_list):
    for text in text_list:
        cleaned = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        X_input = vectorizer.transform([cleaned])
        pred = model.predict(X_input)[0]
        print(f"Text: {text}\nPredicted Sentiment: {pred}\n")

# Example usage
test_texts = [
    "I absolutely love this product, it works perfectly!",  # positive
    "It's fine, nothing extraordinary about it.",  # neutral
    "This is terrible, completely useless and broken!"  # negative
]

test_sentiment(test_texts)


Text: I absolutely love this product, it works perfectly!
Predicted Sentiment: positive

Text: It's fine, nothing extraordinary about it.
Predicted Sentiment: neutral

Text: This is terrible, completely useless and broken!
Predicted Sentiment: negative



In [70]:
joblib.dump((model, vectorizer), "sentiment_model.pkl")
print("Saved sentiment_model.pkl (model + vectorizer)")

Saved sentiment_model.pkl (model + vectorizer)
